In [ ]:
import torchaudio
import soundfile as sf
import torch

def load_audio(path):
    x, sr = sf.read(path)

    x = torch.tensor(x, dtype=torch.float32)

    # stereo → mono
    if len(x.shape) > 1:
        x = x.mean(dim=1)

    # normalization (VERY IMPORTANT)
    x = x / (torch.max(torch.abs(x)) + 1e-6)

    return x

In [ ]:
import torch

def pad_truncate(x, max_len=64000):
    if x.size(0) > max_len:
        return x[:max_len]
    else:
        pad_size = max_len - x.size(0)
        return torch.nn.functional.pad(x, (0, pad_size))

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset

class AudioDataset(Dataset):
    def __init__(self, file_paths, labels):
        self.labels = labels
        self.data = []

        print("Loading dataset into RAM...")

        for path in file_paths:
            x = load_audio(path)   # (T,)

            # convert to tensor
            x = torch.tensor(x, dtype=torch.float32)

            # add channel dimension → (1, T)
            x = x.unsqueeze(0)

            # reduce to fixed size → (1, 100)
            x = F.adaptive_max_pool1d(x, 100)

            self.data.append(x)

        print("Loaded", len(self.data), "samples into RAM")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]